# Data Preparation

In [ ]:
from datasets import load_dataset


data = load_dataset('KrorngAI/KhmerSynthetic1M-FilterOutTooLongSequences',
                    split='train')
data

In [ ]:
data2 = load_dataset('seanghay/khmer-hanuman-100k', split='train')
data2

In [ ]:
data = data.remove_columns(['id', 'file_name'])
data

In [ ]:
from datasets import concatenate_datasets

data = concatenate_datasets([data, data2])
data

In [ ]:
data = data.filter(lambda x: x['image'].width / x['image'].height <= 5, num_proc=2)

In [ ]:
idx = 600
print(data['text'][idx])
data['image'][idx]

In [ ]:
!pip install tror-yong-ocr

In [ ]:
from tror_yong_ocr import get_tokenizer

tokenizer = get_tokenizer()

def preprocess_text(example):
    example['token_ids'] = tokenizer.encode(example['text'], add_special_tokens=True)
    return example

In [ ]:
data = data.map(preprocess_text, remove_columns=['text'], num_proc=2)

In [ ]:
data

In [ ]:
print(tokenizer.decode(data['token_ids'][idx], ignore_special_tokens=True))
data['image'][idx]

In [ ]:
data.save_to_disk('./data')

In [ ]:
!pip install hydra-core==1.3.2 hydra-colorlog==1.2.0 hydra-optuna-sweeper==1.2.0
!pip install lightning>=2.0.0
!pip install torchmetrics>=0.11.4

In [ ]:
!mkdir src

In [ ]:
%%writefile src/data.py
from typing import Any
import lightning as L
import torch
from torchvision.transforms import v2 as transforms
from torch.utils.data import DataLoader
from datasets import load_from_disk
from dataclasses import dataclass


@dataclass
class DataCollatorWithPadding:
    pad_token_id: int
    transform: Any

    def __call__(self, features):
        imgs, inp_tids, tgt_tids = [], [], []
        for feature in features:
            imgs.append(feature['image'])
            inp_tids.append(torch.as_tensor(feature['token_ids'][:-1]))
            tgt_tids.append(torch.as_tensor(feature['token_ids'][1:]))

        return {
            'img_tensor': torch.stack(self.transform(imgs)),
            'input_ids': torch.nn.utils.rnn.pad_sequence(inp_tids, batch_first=True, padding_value=self.pad_token_id),
            'target_ids': torch.nn.utils.rnn.pad_sequence(tgt_tids, batch_first=True, padding_value=self.pad_token_id),
        }


class DataModule(L.LightningDataModule):
    def __init__(self, pad_token_id, data_dir="./data", batch_size=4):
        super().__init__()
        self.save_hyperparameters()

    def setup(self, stage=None):
        self.data = load_from_disk(self.hparams.data_dir)
        self.data = self.data.train_test_split(test_size=0.01, seed=168)
        self.train_transform = transforms.Compose(
            [
                transforms.Resize((32, 128)),
                transforms.ToImage(),
                transforms.ToDtype(torch.float32, scale=True),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        )
        self.val_transform = self.train_transform

    def train_dataloader(self):
        return DataLoader(self.data['train'], batch_size=self.hparams.batch_size,
                          collate_fn=DataCollatorWithPadding(self.hparams.pad_token_id, self.train_transform),
                          shuffle=True
                          )

    def val_dataloader(self):
        return DataLoader(self.data['test'], batch_size=8,
                          collate_fn=DataCollatorWithPadding(self.hparams.pad_token_id, self.val_transform),
                          shuffle=False
                          )

# Train Preparation

In [ ]:
%%writefile train.py
import lightning as L
import torch
import hydra
from omegaconf import DictConfig
from src.data import DataModule
from tror_yong_ocr import TrorYongOCR, TrorYongConfig, get_tokenizer
from torchmetrics import MeanMetric
from torch.optim.lr_scheduler import OneCycleLR
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor, TQDMProgressBar


class TrorYongModule(L.LightningModule):
    def __init__(self, net, lr):
        super().__init__()
        self.save_hyperparameters(ignore=['net'])
        self.net = net
        self.train_loss = MeanMetric()
        self.val_loss = MeanMetric()

    def training_step(self, batch, batch_idx):
        img, input_tokens, target_tokens = batch['img_tensor'], batch['input_ids'], batch['target_ids']
        logits, loss = self.net(img, input_tokens, target_tokens)
        self.train_loss(loss)
        self.log('train/loss', self.train_loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def on_train_start(self):
        self.train_loss.reset()
        self.val_loss.reset()

    def validation_step(self, batch, batch_id):
        img, input_tokens, target_tokens = batch['img_tensor'], batch['input_ids'], batch['target_ids']
        logits, loss = self.net(img, input_tokens, target_tokens)
        self.val_loss(loss)
        self.log('val/loss', self.val_loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.net.parameters(), lr=self.hparams.lr, betas=(0.9, 0.999), weight_decay=0.001)
        scheduler = OneCycleLR(optimizer, max_lr=self.hparams.lr, total_steps=self.trainer.estimated_stepping_batches, pct_start=0.1, anneal_strategy='cos',
                               final_div_factor=20)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}
        }

@hydra.main(version_base="1.3", config_path='configs', config_name='config')
def main(cfg: DictConfig):
    tokenizer = get_tokenizer()

    dm = DataModule(tokenizer.pad_id, **cfg.data)
    config = TrorYongConfig(vocab_size=len(tokenizer), n_channel=3, **cfg.net)
    net = TrorYongOCR(config, tokenizer)
    model = TrorYongModule(net, **cfg.model)

    trainer = L.Trainer(
        enable_progress_bar=True,
        callbacks=[
            TQDMProgressBar(refresh_rate=10),
            LearningRateMonitor(logging_interval='step'),
            ModelCheckpoint(monitor='val/loss')
        ],
        **cfg.trainer
    )

    trainer.fit(model, dm)

if __name__ == "__main__":
    main()

# Configuration

In [ ]:
!mkdir configs

In [ ]:
%%writefile configs/config.yaml

model:
    lr: 0.0007

net:
    img_size: [32, 128]
    patch_size: [4, 8]
    block_size: 192
    n_layer: 4
    n_head: 6
    n_embed: 384
    dropout: 0.1
    bias: true

data:
    data_dir: "./data"
    batch_size: 512


trainer:
    max_epochs: 5
    accelerator: "auto"
    precision: "16-mixed"
    accumulate_grad_batches: 2
    gradient_clip_val: 1.0


In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs/

In [ ]:
!python train.py

# Load Checkpoint

In [ ]:
import torch


state = torch.load("/content/lightning_logs/version_0/checkpoints/epoch=4-step=630.ckpt")

print(state.keys())

In [ ]:
state['state_dict'].keys()

In [ ]:
from tror_yong_ocr import TrorYongOCR, TrorYongConfig
from tror_yong_ocr import get_tokenizer

tokenizer = get_tokenizer()

config = TrorYongConfig(
    img_size=(32, 128),
    patch_size=(4, 8),
    n_channel=3,
    vocab_size=len(tokenizer),
    block_size=192,
    n_layer=4,
    n_head=6,
    n_embed=384,
    dropout=0.1,
    bias=True,
)
net = TrorYongOCR(config, tokenizer)

In [ ]:
net.state_dict().keys()

In [ ]:
new_state_dict = {}
for name, tensor in state['state_dict'].items():
    new_state_dict[name.split('net.')[1]] = tensor
new_state_dict.keys()

In [ ]:
net.load_state_dict(new_state_dict)

In [ ]:
torch.save(net.state_dict(), 'best_model.pt')
net.load_state_dict(torch.load('best_model.pt'))